# 带 Collection Schema 的聊天机器人

## 回顾

我们将聊天机器人扩展为将语义记忆保存到单个[用户资料](https://docs.langchain.com/oss/python/concepts/memory#profile)中。

我们还介绍了 [Trustcall](https://github.com/hinthornw/trustcall) 库，用于用新信息更新该 schema。

## 目标

有时我们希望将记忆保存到[集合](https://docs.google.com/presentation/d/181mvjlgsnxudQI6S3ritg9sooNyu4AcLLFH1UK0kIuk/edit#slide=id.g30eb3c8cf10_0_200)中，而不是单个资料中。

这里我们将更新聊天机器人，[将记忆保存到集合中](https://docs.langchain.com/oss/python/concepts/memory#collection)。

我们还将展示如何使用 Trustcall 更新这个集合。

In [ ]:
%%capture --no-stderr
%pip install -U langchain_deepseek langgraph trustcall langchain_core

In [ ]:
import os, getpass

def _set_env(var: str):
    # 检查变量是否已在 OS 环境变量中设置
    env_value = os.environ.get(var)
    if not env_value:
        # 如果未设置，提示用户输入
        env_value = getpass.getpass(f"{var}: ")
    
    # 为当前进程设置环境变量
    os.environ[var] = env_value

_set_env("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langchain-academy"

## 定义集合 schema

我们将创建一个灵活的集合 schema 来存储关于用户交互的记忆，而不是将用户信息存储在固定的资料结构中。

每条记忆将作为一个独立的条目存储，包含一个 `content` 字段用于存放我们想要记住的主要信息。

这种方法允许我们构建一个开放式的记忆集合，可以随着我们对用户的了解更多而增长和变化。

我们可以将集合 schema 定义为 [Pydantic](https://docs.pydantic.dev/latest/) 对象。

In [ ]:
from pydantic import BaseModel, Field

class Memory(BaseModel):
    content: str = Field(description="The main content of the memory. For example: User expressed interest in learning about French.")

class MemoryCollection(BaseModel):
    memories: list[Memory] = Field(description="A list of memories about the user.")

In [ ]:
_set_env("DEEPSEEK_API_KEY")

我们可以使用 LangChain 的[聊天模型](https://docs.langchain.com/oss/python/langchain/models)接口的 [`with_structured_output`](https://docs.langchain.com/oss/python/langchain/models#structured-outputs) 方法来强制结构化输出。

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek

# 初始化模型
model = ChatDeepSeek(model="deepseek-chat", temperature=0)

# 将 schema 绑定到模型
model_with_structure = model.with_structured_output(MemoryCollection)

# 调用模型，产生与 schema 匹配的结构化输出
memory_collection = model_with_structure.invoke([HumanMessage("My name is Lance. I like to bike.")])
memory_collection.memories

我们可以使用 `model_dump()` 将 Pydantic 模型实例序列化为 Python 字典。

In [ ]:
memory_collection.memories[0].model_dump()

将每条记忆的字典表示保存到存储中。

In [ ]:
import uuid
from langgraph.store.memory import InMemoryStore

# 初始化内存存储
in_memory_store = InMemoryStore()

# 用于保存记忆的命名空间
user_id = "1"
namespace_for_memory = (user_id, "memories")

# 通过键和值将记忆保存到命名空间
key = str(uuid.uuid4())
value = memory_collection.memories[0].model_dump()
in_memory_store.put(namespace_for_memory, key, value)

key = str(uuid.uuid4())
value = memory_collection.memories[1].model_dump()
in_memory_store.put(namespace_for_memory, key, value)

在存储中搜索记忆。

In [ ]:
# 搜索
for m in in_memory_store.search(namespace_for_memory):
    print(m.dict())

## 更新集合 schema

我们在上一课中讨论了更新 profile schema 的挑战。

集合也有同样的问题！

我们希望能够用新记忆更新集合，以及更新集合中的现有记忆。

现在我们将展示 [Trustcall](https://github.com/hinthornw/trustcall) 也可以用来更新集合。

这既可以添加新记忆，也可以[更新集合中的现有记忆](https://github.com/hinthornw/trustcall?tab=readme-ov-file#simultanous-updates--insertions)。

让我们用 Trustcall 定义一个新的提取器。

和之前一样，我们为每条记忆提供 schema，`Memory`。

但是，我们可以提供 `enable_inserts=True` 来允许提取器向集合中插入新记忆。

In [ ]:
from trustcall import create_extractor

# 创建提取器
trustcall_extractor = create_extractor(
    model,
    tools=[Memory],
    tool_choice="Memory",
    enable_inserts=True,
)

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# 指令
instruction = """Extract memories from the following conversation:"""

# 对话
conversation = [HumanMessage(content="Hi, I'm Lance."), 
                AIMessage(content="Nice to meet you, Lance."), 
                HumanMessage(content="This morning I had a nice bike ride in San Francisco.")]

# 调用提取器
result = trustcall_extractor.invoke({"messages": [SystemMessage(content=instruction)] + conversation})

In [ ]:
# 消息包含工具调用
for m in result["messages"]:
    m.pretty_print()

In [ ]:
# 响应包含符合 schema 的记忆
for m in result["responses"]: 
    print(m)

In [ ]:
# 元数据包含工具调用
for m in result["response_metadata"]: 
    print(m)

In [ ]:
# 更新对话
updated_conversation = [AIMessage(content="That's great, did you do after?"), 
                        HumanMessage(content="I went to Tartine and ate a croissant."),                        
                        AIMessage(content="What else is on your mind?"),
                        HumanMessage(content="I was thinking about my Japan, and going back this winter!"),]

# 更新指令
system_msg = """Update existing memories and create new ones based on the following conversation:"""

# 我们将保存现有记忆，赋予它们 ID、键（工具名称）和值
tool_name = "Memory"
existing_memories = [(str(i), tool_name, memory.model_dump()) for i, memory in enumerate(result["responses"])] if result["responses"] else None
existing_memories

In [ ]:
# 使用更新的对话和现有记忆调用提取器
result = trustcall_extractor.invoke({"messages": updated_conversation, 
                                     "existing": existing_memories})

In [ ]:
# 来自模型的消息表明进行了两次工具调用
for m in result["messages"]:
    m.pretty_print()

In [ ]:
# 响应包含符合 schema 的记忆
for m in result["responses"]: 
    print(m)

这告诉我们通过指定 `json_doc_id` 更新了集合中的第一条记忆。

In [ ]:
# 元数据包含工具调用
for m in result["response_metadata"]: 
    print(m)

LangSmith trace: 

https://smith.langchain.com/public/ebc1cb01-f021-4794-80c0-c75d6ea90446/r

## 带集合 schema 更新的聊天机器人

现在，让我们将 Trustcall 引入聊天机器人，来创建和更新记忆集合。

In [ ]:
from IPython.display import Image, display

import uuid

from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.store.memory import InMemoryStore
from langchain_core.messages import merge_message_runs
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.runnables.config import RunnableConfig
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.base import BaseStore

# 初始化模型
model = ChatDeepSeek(model="deepseek-chat", temperature=0)

# 记忆 schema
class Memory(BaseModel):
    content: str = Field(description="The main content of the memory. For example: User expressed interest in learning about French.")

# 创建 Trustcall 提取器
trustcall_extractor = create_extractor(
    model,
    tools=[Memory],
    tool_choice="Memory",
    # 允许提取器插入新记忆
    enable_inserts=True,
)

# 聊天机器人指令
MODEL_SYSTEM_MESSAGE = """You are a helpful chatbot. You are designed to be a companion to a user. 

You have a long term memory which keeps track of information you learn about the user over time.

Current Memory (may include updated memories from this conversation): 

{memory}"""

# Trustcall 指令
TRUSTCALL_INSTRUCTION = """Reflect on following interaction. 

Use the provided tools to retain any necessary memories about the user. 

Use parallel tool calling to handle updates and insertions simultaneously:"""

def call_model(state: MessagesState, config: RunnableConfig, store: BaseStore):

    """从存储中加载记忆，并用其个性化聊天机器人的回答。"""
    
    # 从配置中获取用户 ID
    user_id = config["configurable"]["user_id"]

    # 从存储中检索记忆
    namespace = ("memories", user_id)
    memories = store.search(namespace)

    # 为系统提示格式化记忆
    info = "\n".join(f"- {mem.value['content']}" for mem in memories)
    system_msg = MODEL_SYSTEM_MESSAGE.format(memory=info)

    # 使用记忆和聊天历史进行回答
    response = model.invoke([SystemMessage(content=system_msg)]+state["messages"])

    return {"messages": response}

def write_memory(state: MessagesState, config: RunnableConfig, store: BaseStore):

    """反思聊天历史，并更新记忆集合。"""
    
    # 从配置中获取用户 ID
    user_id = config["configurable"]["user_id"]

    # 定义记忆的命名空间
    namespace = ("memories", user_id)

    # 检索最近的记忆作为上下文
    existing_items = store.search(namespace)

    # 为 Trustcall 提取器格式化现有记忆
    tool_name = "Memory"
    existing_memories = ([(existing_item.key, tool_name, existing_item.value)
                          for existing_item in existing_items]
                          if existing_items
                          else None
                        )

    # 合并聊天历史和指令
    updated_messages=list(merge_message_runs(messages=[SystemMessage(content=TRUSTCALL_INSTRUCTION)] + state["messages"]))

    # 调用提取器
    result = trustcall_extractor.invoke({"messages": updated_messages, 
                                        "existing": existing_memories})

    # 将 Trustcall 产生的记忆保存到存储中
    for r, rmeta in zip(result["responses"], result["response_metadata"]):
        store.put(namespace,
                  rmeta.get("json_doc_id", str(uuid.uuid4())),
                  r.model_dump(mode="json"),
            )

# 定义图
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_node("write_memory", write_memory)
builder.add_edge(START, "call_model")
builder.add_edge("call_model", "write_memory")
builder.add_edge("write_memory", END)

# 用于长期（跨线程）记忆的存储
across_thread_memory = InMemoryStore()

# 用于短期（线程内）记忆的检查点器
within_thread_memory = MemorySaver()

# 使用检查点器和存储编译图
graph = builder.compile(checkpointer=within_thread_memory, store=across_thread_memory)

# 查看
display(Image(graph.get_graph(xray=1).draw_mermaid_png()))

In [ ]:
# 我们提供线程 ID 用于短期（线程内）记忆
# 我们提供用户 ID 用于长期（跨线程）记忆
config = {"configurable": {"thread_id": "1", "user_id": "1"}}

# 用户输入
input_messages = [HumanMessage(content="Hi, my name is Lance")]

# 运行图
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

In [ ]:
# 用户输入
input_messages = [HumanMessage(content="I like to bike around San Francisco")]

# 运行图
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

In [ ]:
# 用于保存记忆的命名空间
user_id = "1"
namespace = ("memories", user_id)
memories = across_thread_memory.search(namespace)
for m in memories:
    print(m.dict())

In [ ]:
# 用户输入
input_messages = [HumanMessage(content="I also enjoy going to bakeries")]

# 运行图
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

在新线程中继续对话。

In [ ]:
# 我们提供线程 ID 用于短期（线程内）记忆
# 我们提供用户 ID 用于长期（跨线程）记忆
config = {"configurable": {"thread_id": "2", "user_id": "1"}}

# 用户输入
input_messages = [HumanMessage(content="What bakeries do you recommend for me?")]

# 运行图
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

### LangSmith

https://smith.langchain.com/public/c87543ec-b426-4a82-a3ab-94d01c01d9f4/r

## Studio

![Screenshot 2024-10-30 at 11.29.25 AM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/6732d0876d3daa19fef993ba_Screenshot%202024-11-11%20at%207.50.21%E2%80%AFPM.png)